# Job Dataset Deduplication

**Problem:** The merged job dataset (1,348 rows from 11 sources) contains duplicates:
- **Within-source:** LinkedIn scraped the same EPAM trainee posting 7 times each
- **Cross-source:** The same job appears on LinkedIn AND the company's own portal (e.g., EPAM, SoftConstruct, Krisp)

**Strategy:**
1. Detect exact duplicates via `full_text` MD5 hash
2. Detect cross-source duplicates via `(job_title, company_name)` matching
3. Keep company portal version over aggregator version
4. Within same source, keep first occurrence

**Input:** `data/processed/jobs/final_jobs_dataset.csv` (1,348 rows)

**Output:** `data/processed/jobs/final_jobs_dataset.csv` (deduplicated, overwrite)

## 1. Load and Inspect

In [1]:
import pandas as pd
import hashlib
from pathlib import Path

BASE = Path.cwd().parent.parent
DATA_PATH = BASE / 'data/processed/jobs/final_jobs_dataset.csv'
BACKUP_PATH = BASE / 'data/processed/jobs/final_jobs_dataset_pre_dedup.csv'

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {len(df)} rows, {len(df.columns)} columns')
print(f'Sources: {df["source"].nunique()}')
print()
print(df['source'].value_counts())

Loaded: 1348 rows, 13 columns
Sources: 11

source
linkedin         992
softconstruct    152
epam             108
staff.am          55
job.am            20
krisp              7
dataart            5
servicetitan       4
picsart            2
synopsys           2
disqo              1
Name: count, dtype: int64


In [2]:
# Add full_text hash for exact content matching
df['_text_hash'] = df['full_text'].astype(str).apply(
    lambda x: hashlib.md5(x.strip().encode('utf-8')).hexdigest()
)

# Add normalized title+company key for cross-source matching
df['_title_norm'] = df['job_title'].astype(str).str.strip().str.lower()
df['_company_norm'] = df['company_name'].astype(str).str.strip().str.lower()

print('Helper columns added.')

Helper columns added.


## 2. Analyze: Within-Source Duplicates

Same source, identical `full_text`. These are scraping artifacts (e.g., LinkedIn returned the same EPAM posting 7 times).

In [3]:
# Find rows with identical full_text within the same source
within_dups = df[df.duplicated(['source', '_text_hash'], keep=False)].copy()
within_groups = within_dups.groupby(['source', '_text_hash'])

print(f'Within-source identical text groups: {within_groups.ngroups}')
print(f'Total rows involved: {len(within_dups)}')
print(f'Rows to remove (keep 1 per group): {len(within_dups) - within_groups.ngroups}')
print()
print('Breakdown by source:')
print(within_dups.groupby('source').size().to_string())

Within-source identical text groups: 28
Total rows involved: 103
Rows to remove (keep 1 per group): 75

Breakdown by source:
source
linkedin    103


In [4]:
# Show the worst offenders
print('Largest within-source duplicate groups:\n')
for (src, h), grp in sorted(within_groups, key=lambda x: -len(x[1]))[:10]:
    print(f'  [{src}] {grp.iloc[0]["job_title"][:60]:60} x{len(grp)} copies')

Largest within-source duplicate groups:

  [linkedin] Automated Testing in JavaScript Trainee                      x7 copies
  [linkedin] Cloud & DevOps Intern at EPAM Lab                            x7 copies
  [linkedin] Software Functional Testing Trainee                          x7 copies
  [linkedin] Automated Testing in Java Trainee                            x7 copies
  [linkedin] Python Development Intern at EPAM Lab                        x7 copies
  [linkedin] Android Development Intern at EPAM Lab                       x7 copies
  [linkedin] .NET Development Trainee                                     x7 copies
  [linkedin] Automated Testing in .NET Trainee                            x7 copies
  [linkedin] Software Functional Testing Intern at EPAM Lab               x7 copies
  [linkedin] Retention Sales Manager                                      x3 copies


## 3. Analyze: Cross-Source Duplicates

Same job posted on both LinkedIn (aggregator) and the company's own career portal. Different `full_text` but same `(job_title, company_name)`.

In [5]:
# Find (title, company) combos that appear in multiple sources
title_company_groups = df.groupby(['_title_norm', '_company_norm'])

cross_source = []
for (title, company), grp in title_company_groups:
    if grp['source'].nunique() > 1:
        cross_source.append({
            'job_title': grp.iloc[0]['job_title'],
            'company_name': grp.iloc[0]['company_name'],
            'sources': sorted(grp['source'].unique().tolist()),
            'source_types': sorted(grp['source_type'].unique().tolist()),
            'count': len(grp),
        })

cross_df = pd.DataFrame(cross_source)
print(f'Cross-source duplicate groups: {len(cross_df)}')
print(f'Total rows involved: {cross_df["count"].sum()}')
print()

Cross-source duplicate groups: 154
Total rows involved: 331



In [6]:
# Which companies appear across sources most?
print('Companies with most cross-source duplicates:\n')
company_counts = cross_df.groupby('company_name').size().sort_values(ascending=False)
for company, count in company_counts.head(10).items():
    print(f'  {company:35} {count} duplicate job pairs')

Companies with most cross-source duplicates:

  EPAM Systems                        99 duplicate job pairs
  SoftConstruct                       45 duplicate job pairs
  Krisp                               7 duplicate job pairs
  CreedRoomz                          1 duplicate job pairs
  DISQO                               1 duplicate job pairs
  Picsart                             1 duplicate job pairs


In [7]:
# Show examples of cross-source duplicates
print('Examples of cross-source duplicates:\n')
for _, row in cross_df.head(15).iterrows():
    print(f'  {row["job_title"][:55]:55} | {row["company_name"]:25} | {row["sources"]}')

Examples of cross-source duplicates:

  .NET Solution Architect with Azure                      | EPAM Systems              | ['epam', 'linkedin']
  .NET/C# Software Engineer                               | SoftConstruct             | ['linkedin', 'softconstruct', 'staff.am']
  2D Artist                                               | SoftConstruct             | ['linkedin', 'softconstruct']
  ABAP Tech Lead                                          | EPAM Systems              | ['epam', 'linkedin']
  Account Manager (English Language)                      | SoftConstruct             | ['linkedin', 'softconstruct']
  Account Manager (Turkish Language)                      | SoftConstruct             | ['linkedin', 'softconstruct']
  Accountant                                              | SoftConstruct             | ['linkedin', 'softconstruct']
  Agile Project Manager                                   | SoftConstruct             | ['linkedin', 'softconstruct']
  AI Automation Engineer

In [8]:
# Which source pairs overlap most?
from collections import Counter

pair_counts = Counter()
for sources in cross_df['sources']:
    for i in range(len(sources)):
        for j in range(i + 1, len(sources)):
            pair_counts[(sources[i], sources[j])] += 1

print('Source overlap matrix:\n')
for (s1, s2), count in pair_counts.most_common():
    print(f'  {s1:15} ↔ {s2:15} : {count} shared jobs')

Source overlap matrix:

  epam            ↔ linkedin        : 99 shared jobs
  linkedin        ↔ softconstruct   : 42 shared jobs
  softconstruct   ↔ staff.am        : 7 shared jobs
  krisp           ↔ linkedin        : 7 shared jobs
  linkedin        ↔ staff.am        : 4 shared jobs
  job.am          ↔ linkedin        : 1 shared jobs
  disqo           ↔ linkedin        : 1 shared jobs
  linkedin        ↔ picsart         : 1 shared jobs


## 4. Summary Before Cleaning

In [9]:
print('=== DUPLICATE SUMMARY ===')
print(f'Total rows before cleaning:        {len(df)}')
print()
print('Within-source (identical full_text):')
n_within = len(within_dups) - within_groups.ngroups
print(f'  Duplicate rows to remove:        {n_within}')
print(f'  Cause: LinkedIn scraped same posting multiple times (EPAM trainee x7 each)')
print()
print('Cross-source (same job, different sources):')
print(f'  Duplicate groups found:          {len(cross_df)}')
print(f'  Rule: keep company_portal, drop aggregator duplicate')

=== DUPLICATE SUMMARY ===
Total rows before cleaning:        1348

Within-source (identical full_text):
  Duplicate rows to remove:        75
  Cause: LinkedIn scraped same posting multiple times (EPAM trainee x7 each)

Cross-source (same job, different sources):
  Duplicate groups found:          154
  Rule: keep company_portal, drop aggregator duplicate


## 5. Deduplication

**Rules (applied in order):**

1. **Within-source exact duplicates:** Same `(source, full_text hash)` → keep first row, drop the rest
2. **Cross-source duplicates:** Same `(job_title_normalized, company_name_normalized)` across sources → keep `company_portal` over `aggregator`. If both are the same type, keep first.

In [10]:
# Backup original
df_original = df.copy()
n_before = len(df)

# ── Step 1: Remove within-source exact text duplicates ─────────────────
before_step1 = len(df)
df = df.drop_duplicates(subset=['source', '_text_hash'], keep='first')
n_within_removed = before_step1 - len(df)
print(f'Step 1 — Within-source text duplicates removed: {n_within_removed}')
print(f'  Rows remaining: {len(df)}')

Step 1 — Within-source text duplicates removed: 75
  Rows remaining: 1273


In [11]:
# ── Step 2: Remove cross-source duplicates ────────────────────────────
# Priority: company_portal > aggregator
# Within same priority: keep first row

SOURCE_TYPE_PRIORITY = {'company_portal': 0, 'aggregator': 1}
df['_priority'] = df['source_type'].map(SOURCE_TYPE_PRIORITY)

before_step2 = len(df)
df = df.sort_values('_priority')  # company_portal first
df = df.drop_duplicates(subset=['_title_norm', '_company_norm'], keep='first')
n_cross_removed = before_step2 - len(df)
print(f'Step 2 — Cross-source duplicates removed: {n_cross_removed}')
print(f'  Rows remaining: {len(df)}')

Step 2 — Cross-source duplicates removed: 205
  Rows remaining: 1068


/var/folders/0l/7b8_ztcx19d3zxdnsypbsg1r0000gn/T/ipykernel_40595/436404577.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['_priority'] = df['source_type'].map(SOURCE_TYPE_PRIORITY)


In [12]:
# Clean up helper columns
df = df.drop(columns=['_text_hash', '_title_norm', '_company_norm', '_priority'])
df = df.sort_values(['source_type', 'source', 'company_name', 'job_title']).reset_index(drop=True)

n_after = len(df)
print(f'\n=== DEDUPLICATION COMPLETE ===')
print(f'Before:                     {n_before}')
print(f'Within-source removed:      {n_within_removed}')
print(f'Cross-source removed:       {n_cross_removed}')
print(f'Total removed:              {n_before - n_after}')
print(f'After:                      {n_after}')


=== DEDUPLICATION COMPLETE ===
Before:                     1348
Within-source removed:      75
Cross-source removed:       205
Total removed:              280
After:                      1068


## 6. Verify Cleaned Dataset

In [13]:
print('Source distribution after dedup:\n')
print(df['source'].value_counts().to_string())
print()
print('Source type distribution:')
print(df['source_type'].value_counts().to_string())

Source distribution after dedup:

source
linkedin         734
softconstruct    141
epam             104
staff.am          48
job.am            20
krisp              7
dataart            5
servicetitan       4
picsart            2
synopsys           2
disqo              1

Source type distribution:
source_type
aggregator        802
company_portal    266


In [14]:
# Verify no remaining duplicates
hash_col = df['full_text'].astype(str).apply(lambda x: hashlib.md5(x.strip().encode()).hexdigest())
remaining_text_dups = hash_col.duplicated().sum()

title_comp = df[['job_title','company_name']].apply(lambda r: (r[0].strip().lower(), r[1].strip().lower()), axis=1)
remaining_tc_dups = title_comp.duplicated().sum()

print(f'Remaining full_text duplicates:         {remaining_text_dups}')
print(f'Remaining (title, company) duplicates:  {remaining_tc_dups}')
print()
if remaining_text_dups == 0 and remaining_tc_dups == 0:
    print('Dataset is clean — no duplicates remain.')
else:
    print('NOTE: Some (title, company) pairs may remain if they are genuinely different jobs')
    print('      (e.g., same title reposted months apart with different descriptions).')

Remaining full_text duplicates:         0
Remaining (title, company) duplicates:  0

Dataset is clean — no duplicates remain.


/var/folders/0l/7b8_ztcx19d3zxdnsypbsg1r0000gn/T/ipykernel_40595/2867558346.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  title_comp = df[['job_title','company_name']].apply(lambda r: (r[0].strip().lower(), r[1].strip().lower()), axis=1)


In [15]:
# Field coverage after dedup
print('Field coverage:\n')
for col in df.columns:
    filled = df[col].astype(str).str.strip().ne('').sum()
    pct = filled / len(df) * 100
    if pct < 100:
        print(f'  {col:20} {filled:>5}/{len(df)} ({pct:.0f}%)')
    else:
        print(f'  {col:20} {filled:>5}/{len(df)} (100%)')

Field coverage:

  source                1068/1068 (100%)
  source_type           1068/1068 (100%)
  source_url            1068/1068 (100%)
  job_title             1068/1068 (100%)
  company_name          1068/1068 (100%)
  location              1068/1068 (100%)
  employment_type       1068/1068 (100%)
  seniority_level       1068/1068 (100%)
  industries            1068/1068 (100%)
  posting_date          1068/1068 (100%)
  deadline              1068/1068 (100%)
  skills_tags           1068/1068 (100%)
  full_text             1068/1068 (100%)


## 7. What Was Removed — Examples

In [16]:
# Identify which rows were removed
kept_idx = set(df.index)
removed = df_original[~df_original.index.isin(kept_idx)].copy()

# Categorize removed rows
removed_hash = df_original[df_original.duplicated(['source', '_text_hash'], keep='first')]
# Cross-source removed = the rest

print(f'Total rows removed: {len(removed)}')
print(f'\n--- Within-source duplicates removed (sample) ---\n')
for _, r in removed_hash.head(8).iterrows():
    print(f'  [{r["source"]:15}] {r["job_title"][:65]}')

print(f'\n--- Cross-source duplicates removed (sample) ---')
print(f'    (aggregator version dropped in favor of company_portal)\n')
# Show examples where aggregator was dropped
cross_removed = removed[~removed.index.isin(removed_hash.index)]
for _, r in cross_removed[cross_removed['source_type'] == 'aggregator'].head(10).iterrows():
    print(f'  DROPPED [{r["source"]:10}] {r["job_title"][:50]:50} @ {r["company_name"]}')

Total rows removed: 280

--- Within-source duplicates removed (sample) ---

  [linkedin       ] Content Research (entry level)
  [linkedin       ] Data Center Technician - AM - Getazat - On-site
  [linkedin       ] Data Center Technician - AM - Abovyan - On-site
  [linkedin       ] Data Center Technician - AM - Bagratashen - On-site
  [linkedin       ] Python Development Intern at EPAM Lab
  [linkedin       ] Python Development Intern at EPAM Lab
  [linkedin       ] Python Development Intern at EPAM Lab
  [linkedin       ] Python Development Intern at EPAM Lab

--- Cross-source duplicates removed (sample) ---
    (aggregator version dropped in favor of company_portal)



## 8. Save

In [17]:
# Save backup of pre-dedup dataset
df_original.drop(columns=['_text_hash', '_title_norm', '_company_norm'], errors='ignore').to_csv(
    BACKUP_PATH, index=False, encoding='utf-8'
)
print(f'Backup saved: {BACKUP_PATH.name} ({len(df_original)} rows)')

# Overwrite with cleaned version
df.to_csv(DATA_PATH, index=False, encoding='utf-8')
print(f'Cleaned saved: {DATA_PATH.name} ({len(df)} rows)')

Backup saved: final_jobs_dataset_pre_dedup.csv (1348 rows)


Cleaned saved: final_jobs_dataset.csv (1068 rows)


## 9. Summary for Thesis

This cell prints a paragraph suitable for inclusion in Chapter 4 (Methodology).

In [18]:
print(f'''DEDUPLICATION SUMMARY (for Chapter 4)
─────────────────────────────────────
The initial merged dataset contained {n_before} job postings from 11 sources.
Two categories of duplicates were identified and removed:

1. Within-source duplicates ({n_within_removed} rows removed): identical job
   descriptions appearing multiple times within the same source, caused by
   scraping artifacts (e.g., EPAM trainee postings duplicated up to 7 times
   each in the LinkedIn scrape). Detected via MD5 hash of full_text.

2. Cross-source duplicates ({n_cross_removed} rows removed): the same job
   appearing on both an aggregator (LinkedIn, Staff.am) and the company's
   own career portal. Detected via normalized (job_title, company_name)
   matching. In all cases, the company portal version was retained over
   the aggregator version, as company portals represent direct employer
   demand with typically richer job descriptions.

After deduplication: {n_after} unique job postings.
The pre-dedup dataset is archived as final_jobs_dataset_pre_dedup.csv.
''')

DEDUPLICATION SUMMARY (for Chapter 4)
─────────────────────────────────────
The initial merged dataset contained 1348 job postings from 11 sources.
Two categories of duplicates were identified and removed:

1. Within-source duplicates (75 rows removed): identical job
   descriptions appearing multiple times within the same source, caused by
   scraping artifacts (e.g., EPAM trainee postings duplicated up to 7 times
   each in the LinkedIn scrape). Detected via MD5 hash of full_text.

2. Cross-source duplicates (205 rows removed): the same job
   appearing on both an aggregator (LinkedIn, Staff.am) and the company's
   own career portal. Detected via normalized (job_title, company_name)
   matching. In all cases, the company portal version was retained over
   the aggregator version, as company portals represent direct employer
   demand with typically richer job descriptions.

After deduplication: 1068 unique job postings.
The pre-dedup dataset is archived as final_jobs_dataset_pre_ded